# 02 · Treino

Treina o modelo de previsão **1X2** (vitória do mandante / empate / vitória do visitante) a partir das features do `01` e grava `models/model.joblib`.

### Contrato
**Entrada:** `data/processed/features.parquet` (linhas `split=="train"`).

**Saída:** `models/model.joblib` — um bundle `{"pipeline", "meta"}`; o
`pipeline` é um `Pipeline` sklearn **fitted** que mapeia as 22 features para
probabilidades 1X2. O `03` só faz `joblib.load(...).predict_proba(X)`.

Plano: [`docs/train/README.md`](../docs/train/README.md) (§8 features, §10
arquitetura, §11 riscos). **Otimizado para o RPS** da skill de avaliação;
**nunca** lê `worldcup-2026-results.csv` (gabarito).

## E0 · Setup, carga e separação `train`/`predict`

Resolve a raiz, carrega `features.parquet`, separa `train` (o `predict` é do
`03` — **não tocar**) e congela a lista de 22 preditores (§8).

In [23]:
# Setup: resolve a raiz do repositório e os caminhos de dados.
from pathlib import Path

def find_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for cand in [base, *base.parents]:
        if (cand / "data" / "raw").is_dir() and (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError("Raiz do repositório não encontrada a partir de " + str(base))

ROOT = find_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
MODELS = ROOT / "models"
print("ROOT:", ROOT)
MODELS.mkdir(parents=True, exist_ok=True)

ROOT: /Users/franklaercio/Workspace/paul-the-octopus


In [24]:
import warnings
from datetime import datetime, timezone

import joblib
import matplotlib
import numpy as np
import pandas as pd
import sklearn

matplotlib.use("Agg")  # backend não-interativo (notebook executado headless no CI)
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.base import clone  # noqa: E402
from sklearn.calibration import CalibratedClassifierCV  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import (  # noqa: E402
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.impute import SimpleImputer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import (  # noqa: E402
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

SEED = 42
np.random.seed(SEED)

# Figuras versionadas (registro, como a EDA): tabela de modelos, calibração, curva alpha.
FIG_DIR = ROOT / "docs" / "train" / "figuras"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("sklearn", sklearn.__version__)

sklearn 1.7.1


In [25]:
# Carrega o artefato do 01 e separa o split de treino (predict é do 03).
features = pd.read_parquet(PROCESSED / "features.parquet")
train = features[features["split"] == "train"].reset_index(drop=True)
predict = features[features["split"] == "predict"].reset_index(drop=True)
print("train:", train.shape, "| predict:", predict.shape)

train: (49400, 31) | predict: (72, 31)


In [26]:
# Lista CONGELADA de preditores (§8) — fonte única; vira meta['feature_names'].
# As 22 colunas que entram como X; as 9 restantes (chaves/alvo/peso) ficam fora.
FEATURE_NAMES = [
    "is_neutral", "confed_home", "confed_away",
    "elo_home", "elo_away", "elo_diff",
    "rank_home", "rank_away", "rank_diff", "points_diff",
    "form_pts_home", "form_pts_away", "form_gf_home", "form_gf_away",
    "form_ga_home", "form_ga_away",
    "rest_days_home", "rest_days_away",
    "h2h_games", "h2h_winrate_home", "h2h_available", "tournament_tier",
]  # 22 — ordem == ordem em meta['feature_names']

# Subgrupos derivados por DIFERENÇA DE CONJUNTOS (não redigitar — R6/§10.1).
CAT_FEATURES = ["confed_home", "confed_away"]          # string nullable
BOOL_FEATURES = ["is_neutral", "h2h_available"]         # boolean nullable
NUM_FEATURES = [c for c in FEATURE_NAMES if c not in CAT_FEATURES + BOOL_FEATURES]
assert set(NUM_FEATURES + CAT_FEATURES + BOOL_FEATURES) == set(FEATURE_NAMES)
assert len(FEATURE_NAMES) == 22

# X/y/w/dates do treino. NÃO fazer .astype() fora do Pipeline: o sklearn consome
# os dtypes nullable (boolean/string) do parquet cru (verificado — R9).
X = train[FEATURE_NAMES]
y = train["target_outcome"].astype("object")    # rótulos home/draw/away
w = train["sample_weight"]                        # peso de treino (só no fit)
dates = train["date"]                             # eixo do split temporal (NÃO é feature)
print("X:", X.shape, "| y classes:", sorted(y.unique()))

X: (49400, 22) | y classes: ['away', 'draw', 'home']


## E1 · Harness de avaliação (RPS + baselines + walk-forward)

**A régua primeiro** (antes de treinar): uma rotina única que pontua qualquer
modelo do **mesmo jeito que a skill**, para a comparação ser justa e o número
do `02` casar com a avaliação final.

Guardrails (§7.D / §10.4):

- **Walk-forward** (`TimeSeriesSplit` sobre o `train` ordenado por data) —
  nunca K-fold aleatório (vazaria o futuro no treino).
- Pré-processamento **dentro** do `Pipeline` → `fit` só no treino do fold.
- `sample_weight` no `.fit` via `clf__sample_weight`; **avaliação SEM peso**
  (imita os 72 jogos de produção, que a skill pontua sem peso).
- `predict_proba` reordenado **por nome** (`classes_` é alfabético
  `away,draw,home`; o RPS exige `home,draw,away`) — R1.

In [27]:
# Métricas REPLICADAS da skill (mesma matemática, mesma ordem OUTCOMES), para o
# número do 02 casar bit-a-bit com a avaliação final (test de paridade em
# tests/test_train.py). NÃO importamos do score_predictions aqui de propósito —
# replicar mantém o notebook desacoplado da skill (e nunca toca o gabarito).
OUTCOMES = ("home", "draw", "away")
EPS = 1e-15

def rps_mean(probs: np.ndarray, onehots: np.ndarray) -> float:
    """Ranked Probability Score médio (saídas ordinais). Menor = melhor."""
    cdf_p = np.cumsum(probs, axis=1)
    cdf_o = np.cumsum(onehots, axis=1)
    r = probs.shape[1]
    return float(np.mean(np.sum((cdf_p[:, :-1] - cdf_o[:, :-1]) ** 2, axis=1) / (r - 1)))

def brier_multiclass(probs: np.ndarray, onehots: np.ndarray) -> float:
    return float(np.mean(np.sum((probs - onehots) ** 2, axis=1)))

def log_loss_mean(probs: np.ndarray, onehots: np.ndarray) -> float:
    clipped = np.clip(probs, EPS, 1.0)
    return float(np.mean(-np.sum(onehots * np.log(clipped), axis=1)))

def accuracy(probs: np.ndarray, actual_idx: np.ndarray) -> float:
    return float(np.mean(np.argmax(probs, axis=1) == actual_idx))

In [28]:
def onehot(labels) -> np.ndarray:
    """Rótulos home/draw/away -> matriz one-hot na ordem OUTCOMES."""
    labels = list(labels)
    idx = np.array([OUTCOMES.index(o) for o in labels])
    oh = np.zeros((len(labels), 3))
    oh[np.arange(len(labels)), idx] = 1.0
    return oh

def proba_in_outcome_order(fitted, X_in) -> np.ndarray:
    """Reordena predict_proba de classes_ (alfabético) -> OUTCOMES (ordem do RPS).

    Função ÚNICA reusada no harness, no 03 e na validate_model — centralizar
    evita três mapeamentos divergentes (R1). classes_ vem de np.unique
    (away, draw, home); o RPS exige (home, draw, away).
    """
    cols = [list(fitted.classes_).index(o) for o in OUTCOMES]
    return fitted.predict_proba(X_in)[:, cols]

def apply_shrinkage(proba: np.ndarray, base_rate, alpha: float) -> np.ndarray:
    """Encolhe as probabilidades para a taxa-base: P' = (1-alpha)*P + alpha*base.

    Regularizador honesto contra "modelo confiante erra" (déficit de empate em
    jogos parelhos / excesso de empates por azar de amostra). `base_rate` é o
    vetor de frequências de classe na ordem OUTCOMES (home/draw/away). Com
    alpha=0 reproduz `proba` EXATAMENTE (cru); preserva o simplex (combinação
    convexa de dois vetores que já somam 1). Função ÚNICA reusada no harness,
    no 03 e na validate_model — o mesmo ajuste em treino, validação e produção.
    """
    if alpha == 0.0:
        return proba
    base = np.asarray(base_rate, dtype=float).reshape(1, -1)
    return (1.0 - alpha) * proba + alpha * base

In [29]:
# Ordena o train por data UMA vez, fora do loop de folds (mergesort = estável:
# empates de data mantêm a ordem (date, home) do artefato → determinístico). O
# TimeSeriesSplit pressupõe ordem temporal; não reordenar é o jeito fácil de vazar.
_order = np.argsort(dates.to_numpy(), kind="mergesort")
X_sorted = X.iloc[_order].reset_index(drop=True)
y_sorted = y.iloc[_order].reset_index(drop=True)
w_sorted = w.iloc[_order].reset_index(drop=True)
dates_sorted = dates.iloc[_order].reset_index(drop=True)

def _slice_since(date_min):
    """Máscara booleana do train ordenado a partir de date_min (ou tudo)."""
    if date_min is None:
        return np.ones(len(dates_sorted), dtype=bool)
    return dates_sorted.to_numpy() >= np.datetime64(date_min)

In [30]:
def evaluate(estimator, n_splits=5, date_min=None):
    """RPS/Brier/log-loss/acc médios em walk-forward, leak-free (§10.4).

    clone por fold (estado fitado não contamina o próximo); sample_weight no fit
    via clf__sample_weight; avaliação SEM peso. date_min recorta a fatia recente
    (ex.: >=1993, a mais parecida com 2026). Retorna (média_entre_folds, por_fold).
    """
    mask = _slice_since(date_min)
    Xs, ys, ws = X_sorted[mask], y_sorted[mask], w_sorted[mask]
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for tr_idx, te_idx in tscv.split(Xs):
        model = clone(estimator)
        model.fit(
            Xs.iloc[tr_idx], ys.iloc[tr_idx],
            clf__sample_weight=ws.iloc[tr_idx].to_numpy(dtype=float),
        )
        proba = proba_in_outcome_order(model, Xs.iloc[te_idx])
        oh = onehot(ys.iloc[te_idx])
        rows.append({                              # avaliação SEM peso (§7.D)
            "rps": rps_mean(proba, oh),
            "brier": brier_multiclass(proba, oh),
            "log_loss": log_loss_mean(proba, oh),
            "acc": accuracy(proba, np.argmax(oh, axis=1)),
        })
    per_fold = pd.DataFrame(rows)
    return per_fold.mean().to_dict(), per_fold

In [31]:
# Baselines no MESMO split (§6.1). A taxa-base é estimada SÓ com o y do treino do
# fold (estimá-la no histórico inteiro vazaria o futuro do fold). Sempre-mandante
# e maior-ranking são determinísticos por jogo → comparam por acurácia (como a
# skill); para o RPS o baseline probabilístico honesto é a taxa-base.
def evaluate_baselines(n_splits=5, date_min=None):
    mask = _slice_since(date_min)
    Xs, ys = X_sorted[mask], y_sorted[mask]
    tscv = TimeSeriesSplit(n_splits=n_splits)
    base, home, rankacc = [], [], []
    for tr_idx, te_idx in tscv.split(Xs):
        y_tr, y_te = ys.iloc[tr_idx], ys.iloc[te_idx]
        oh = onehot(y_te)
        # taxa-base: frequências 1X2 do treino do fold, broadcast no teste
        rate = y_tr.value_counts(normalize=True).reindex(OUTCOMES).fillna(0.0).to_numpy()
        base.append(rps_mean(np.tile(rate, (len(y_te), 1)), oh))
        # sempre-mandante: acurácia = fração de jogos com resultado 'home'
        home.append(float(np.mean(y_te.to_numpy() == "home")))
        # maior-ranking: prevê 'home' se rank_home < rank_away (menor = melhor)
        rh = Xs["rank_home"].iloc[te_idx].to_numpy(dtype=float)
        ra = Xs["rank_away"].iloc[te_idx].to_numpy(dtype=float)
        valid = ~np.isnan(rh) & ~np.isnan(ra) & (rh != ra)
        if valid.any():
            pick = np.where(rh[valid] < ra[valid], "home", "away")
            rankacc.append(float(np.mean(pick == y_te.to_numpy()[valid])))
    return {
        "taxa_base_rps": float(np.mean(base)),
        "sempre_mandante_acc": float(np.mean(home)),
        "maior_ranking_acc": float(np.mean(rankacc)) if rankacc else float("nan"),
    }

In [32]:
# Coleta, por dobra walk-forward, (proba_crua, base_da_dobra, onehot) para tunar o
# shrinkage (E6.5) reusando OS MESMOS folds — treina uma vez por fold e varre alpha
# em cima, sem re-treinar. A taxa-base é a do TREINO DA DOBRA (leak-free): estimá-la
# no histórico inteiro vazaria o futuro do fold. Mesmo split/peso/avaliação-sem-peso
# do evaluate (§10.4). Usado SÓ para escolher alpha; nunca toca os 20 jogos reais.
def collect_folds(estimator, n_splits=5, date_min=None):
    mask = _slice_since(date_min)
    Xs, ys, ws = X_sorted[mask], y_sorted[mask], w_sorted[mask]
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []
    for tr_idx, te_idx in tscv.split(Xs):
        model = clone(estimator)
        model.fit(
            Xs.iloc[tr_idx], ys.iloc[tr_idx],
            clf__sample_weight=ws.iloc[tr_idx].to_numpy(dtype=float),
        )
        proba = proba_in_outcome_order(model, Xs.iloc[te_idx])
        oh = onehot(ys.iloc[te_idx])
        rate = (
            ys.iloc[tr_idx].value_counts(normalize=True)
            .reindex(OUTCOMES).fillna(0.0).to_numpy()
        )
        folds.append((proba, rate, oh))
    return folds

def rps_shrunk_over_folds(folds, alpha):
    """RPS médio entre folds aplicando apply_shrinkage (base por dobra) — leak-free."""
    return float(np.mean([
        rps_mean(apply_shrinkage(proba, rate, alpha), oh) for proba, rate, oh in folds
    ]))

## E2 · Definição do alvo

**Decisão: classificador 1X2 direto** (3 classes ordinais `home/draw/away`).
O `01` já entrega `target_outcome` pronto; otimiza direto o que vira RPS; baixo
risco. A rota de **modelo de gols (Poisson/Dixon-Coles)** fica registrada como
alternativa medível (placares estão no artefato) — só valeria o esforço se o
1X2 calibrado perdesse para a taxa-base na fatia de jogos equilibrados (não é o
caso aqui, ver E4). Mantemos 1X2.

## E3 · Pré-processamento  ·  E4 · Baseline forte (LogisticRegression)

Cada candidato leva **seu próprio** pré-processamento **dentro** do `Pipeline`
(linear precisa de imputação+scaler; HistGB não) — a comparação continua justa
porque o harness avalia todos no mesmo split.

**Linear:** `SimpleImputer(median, add_indicator=True)` (o indicador materializa
o *sinal de época* dos 41,5% de ranking ausente pré-1993) + `StandardScaler` nas
numéricas; `OneHotEncoder(handle_unknown="ignore")` nas `confed_*`; booleanas
passthrough. `keep_empty_features=True` evita um `UserWarning` quando um fold de
treino é 100% pré-1993 (coluna de ranking toda-NaN) — o indicador de ausência
ainda é emitido. **Sem** `multi_class` (removido na 1.7 — R2; multinomial é o
default). `max_iter=1000` evita `ConvergenceWarning`.

In [33]:
# Pré-processamento do pipeline LINEAR (dentro do Pipeline → fit só no fold).
num_lin = Pipeline([
    # +indicador de ausência (sinal pré-1993); keep_empty_features silencia o
    # UserWarning de coluna toda-NaN num fold antigo (R11, pipeline sem warning).
    ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("scaler", StandardScaler()),
])
pre_lin = ColumnTransformer([
    ("num", num_lin, NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
    ("bool", "passthrough", BOOL_FEATURES),   # boolean nullable -> sklearn trata como 0/1
])
pipe_lin = Pipeline([
    ("pre", pre_lin),
    # SEM multi_class (removido na 1.7 — R2); multinomial é o default em multiclasse.
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

In [34]:
# Avalia a logística no harness e compara com os 3 baselines (global e >=1993).
# O CI roda com warnings; garantimos que o pipeline não cospe nenhum.
with warnings.catch_warnings():
    warnings.simplefilter("error")
    lin_global, lin_global_folds = evaluate(pipe_lin)
    lin_1993, _ = evaluate(pipe_lin, date_min="1993-01-01")
base_global = evaluate_baselines()
base_1993 = evaluate_baselines(date_min="1993-01-01")
print("logreg  global :", {k: round(v, 5) for k, v in lin_global.items()})
print("logreg  >=1993 :", {k: round(v, 5) for k, v in lin_1993.items()})
print("baselines global:", {k: round(v, 5) for k, v in base_global.items()})
print("baselines >=1993:", {k: round(v, 5) for k, v in base_1993.items()})

logreg  global : {'rps': 0.17539, 'brier': 0.52578, 'log_loss': 0.89238, 'acc': 0.58747}
logreg  >=1993 : {'rps': 0.17252, 'brier': 0.5175, 'log_loss': 0.8802, 'acc': 0.59335}
baselines global: {'taxa_base_rps': 0.22562, 'sempre_mandante_acc': 0.48493, 'maior_ranking_acc': 0.5532}
baselines >=1993: {'taxa_base_rps': 0.22691, 'sempre_mandante_acc': 0.48164, 'maior_ranking_acc': 0.55764}


In [35]:
# class_weight='balanced' — testar se ajuda o RPS (empate é a classe minoritária).
# Manter só se melhorar; senão fica registrado que PIORA (distorce calibração).
pipe_lin_balanced = Pipeline([
    ("pre", pre_lin),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")),
])
with warnings.catch_warnings():
    warnings.simplefilter("error")
    bal_global, _ = evaluate(pipe_lin_balanced)
    bal_1993, _ = evaluate(pipe_lin_balanced, date_min="1993-01-01")
print("logreg balanced global:", {k: round(v, 5) for k, v in bal_global.items()})
print("logreg balanced >=1993:", {k: round(v, 5) for k, v in bal_1993.items()})
print(
    "-> class_weight ajuda RPS (>=1993)?",
    bal_1993["rps"] < lin_1993["rps"],
    "| mantendo o default (sem class_weight).",
)

logreg balanced global: {'rps': 0.18362, 'brier': 0.55127, 'log_loss': 0.93175, 'acc': 0.54811}
logreg balanced >=1993: {'rps': 0.17958, 'brier': 0.54149, 'log_loss': 0.9189, 'acc': 0.55804}
-> class_weight ajuda RPS (>=1993)? False | mantendo o default (sem class_weight).


## E5 · Alternativas — HistGradientBoostingClassifier

Boosting nativo do sklearn (sem custo de dependência): captura não-linearidades
e interações (ex.: `is_neutral` × `elo_diff`), **aceita NaN nativamente**
(resolve os 41,5% sem ranking sem imputar) e dispensa scaler. Usamos a variante
com **categóricas nativas** (`categorical_features="from_dtype"`), com a
conversão para `category` **dentro** do `Pipeline` (sem train/serve skew — R9).

In [36]:
# Variante recomendada: confed_* como category NATIVA (conversão dentro do Pipeline).
to_category = FunctionTransformer(
    lambda Z: Z.assign(**{c: Z[c].astype("category") for c in CAT_FEATURES})
)
pipe_hgb = Pipeline([
    ("to_cat", to_category),          # dentro do Pipeline → sem skew
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED, categorical_features="from_dtype")),
])
with warnings.catch_warnings():
    warnings.simplefilter("error")
    hgb_global, _ = evaluate(pipe_hgb)
    hgb_1993, _ = evaluate(pipe_hgb, date_min="1993-01-01")
print("histgb global :", {k: round(v, 5) for k, v in hgb_global.items()})
print("histgb >=1993 :", {k: round(v, 5) for k, v in hgb_1993.items()})

histgb global : {'rps': 0.18132, 'brier': 0.54386, 'log_loss': 0.92801, 'acc': 0.57287}
histgb >=1993 : {'rps': 0.17861, 'brier': 0.53535, 'log_loss': 0.91756, 'acc': 0.57955}


In [37]:
# Tabela comparativa: modelo × {RPS, Brier, log-loss, acc} vs. os 3 baselines.
# A REGRA DURA: o modelo escolhido tem de vencer a taxa-base em RPS.
results = pd.DataFrame(
    [
        {"modelo": "taxa_base", "rps": base_global["taxa_base_rps"],
         "rps_1993": base_1993["taxa_base_rps"], "brier": np.nan,
         "log_loss": np.nan, "acc": np.nan},
        {"modelo": "sempre_mandante", "rps": np.nan, "rps_1993": np.nan,
         "brier": np.nan, "log_loss": np.nan, "acc": base_global["sempre_mandante_acc"]},
        {"modelo": "maior_ranking", "rps": np.nan, "rps_1993": np.nan,
         "brier": np.nan, "log_loss": np.nan, "acc": base_global["maior_ranking_acc"]},
        {"modelo": "logreg", "rps": lin_global["rps"], "rps_1993": lin_1993["rps"],
         "brier": lin_global["brier"], "log_loss": lin_global["log_loss"],
         "acc": lin_global["acc"]},
        {"modelo": "histgb", "rps": hgb_global["rps"], "rps_1993": hgb_1993["rps"],
         "brier": hgb_global["brier"], "log_loss": hgb_global["log_loss"],
         "acc": hgb_global["acc"]},
    ]
).set_index("modelo")
results.round(5)

,rps,rps_1993,brier,log_loss,acc
modelo,,,,,
taxa_base,0.22562,0.22691,NaN,NaN,NaN
sempre_mandante,NaN,NaN,NaN,NaN,0.48493
maior_ranking,NaN,NaN,NaN,NaN,0.55320
logreg,0.17539,0.17252,0.52578,0.89238,0.58747
histgb,0.18132,0.17861,0.54386,0.92801,0.57287


## E6 · Calibração, seleção, ajuste final e persistência

1. **Seleção** pelo menor **RPS de validação**, priorizando a **fatia ≥1993**
   (a mais parecida com 2026 — nota de engenharia §11); desempate por calibração,
   depois parcimônia (modelo mais simples vence empate técnico).
2. **Calibração** do escolhido (`CalibratedClassifierCV`, `cv` **temporal**):
   medir RPS antes/depois e **manter só se melhorar por delta** (o leak parcial
   da calibração interna torna o absoluto otimista — R3; decidir pelo delta).
3. **Re-treinar** o pipeline escolhido em **todo** o `train` (com `sample_weight`).
4. **Persistir** o bundle `{"pipeline", "meta"}` em `models/model.joblib`.

In [38]:
# Seleção: candidatos por RPS (prioriza fatia >=1993). Parcimônia desempata.
candidates = {
    "logreg": {"pipe": pipe_lin, "rps_1993": lin_1993["rps"], "rps_global": lin_global["rps"]},
    "histgb": {"pipe": pipe_hgb, "rps_1993": hgb_1993["rps"], "rps_global": hgb_global["rps"]},
}
best_name = min(candidates, key=lambda k: candidates[k]["rps_1993"])
best_pipe = candidates[best_name]["pipe"]
print("Modelo escolhido:", best_name)
print("  RPS >=1993:", round(candidates[best_name]["rps_1993"], 5),
      "| RPS global:", round(candidates[best_name]["rps_global"], 5))
print("  taxa-base >=1993:", round(base_1993["taxa_base_rps"], 5),
      "| vence a taxa-base?", candidates[best_name]["rps_1993"] < base_1993["taxa_base_rps"])

Modelo escolhido: logreg
  RPS >=1993: 0.17252 | RPS global: 0.17539
  taxa-base >=1993: 0.22691 | vence a taxa-base? True


In [39]:
# Calibração temporal: isotônica (temos ~28k jogos >=1993) e sigmoide. Decidir
# por DELTA de RPS (R3); cv=TimeSeriesSplit — nunca cv=int (embaralha) nem
# cv='prefit' (calibraria no mesmo dado do fit).
cal_iso = CalibratedClassifierCV(best_pipe, method="isotonic", cv=TimeSeriesSplit(5))
cal_sig = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=TimeSeriesSplit(5))
with warnings.catch_warnings():
    warnings.simplefilter("error")
    iso_1993, _ = evaluate(cal_iso, date_min="1993-01-01")
    sig_1993, _ = evaluate(cal_sig, date_min="1993-01-01")
base_rps_1993 = candidates[best_name]["rps_1993"]
calib_table = pd.DataFrame({
    "rps_1993": [base_rps_1993, iso_1993["rps"], sig_1993["rps"]],
}, index=["sem_calibracao", "isotonic", "sigmoid"]).round(5)
print(calib_table)

# Mantém a calibração só se melhorar o RPS (delta). Senão, fica o pipeline cru.
best_calib = min([("isotonic", iso_1993["rps"], cal_iso),
                  ("sigmoid", sig_1993["rps"], cal_sig)], key=lambda t: t[1])
use_calibration = best_calib[1] < base_rps_1993
if use_calibration:
    final_estimator = best_calib[2]
    model_name = f"{best_name}_{best_calib[0]}"
    cv_rps_final = best_calib[1]
    print(f"-> calibração MANTIDA ({best_calib[0]}): {best_calib[1]:.5f} < {base_rps_1993:.5f}")
else:
    final_estimator = best_pipe
    model_name = best_name
    cv_rps_final = base_rps_1993
    print(f"-> calibração DESCARTADA (não melhora o RPS; {best_calib[0]} "
          f"{best_calib[1]:.5f} >= {base_rps_1993:.5f}). Mantendo o pipeline cru.")

                rps_1993
sem_calibracao   0.17252
isotonic         0.17870
sigmoid          0.17916
-> calibração DESCARTADA (não melhora o RPS; isotonic 0.17870 >= 0.17252). Mantendo o pipeline cru.


## E6.5 · Shrinkage para a taxa-base (`P' = (1−α)·P + α·base`)

Regularizador honesto contra o único modo de falha observado nos jogos reais
(palpite confiante que bate em empate / déficit de empate em jogos parelhos):
encolher as probabilidades em direção às frequências de classe. **α é ajustado
SOMENTE no walk-forward** — busca em grade `α ∈ {0, 0.05, …, 0.5}` escolhendo o
que minimiza o **RPS médio walk-forward** (avaliação sem peso, mesmo harness),
com a `base` de cada dobra estimada **no treino da dobra** (leak-free). **Nunca**
usamos os 20 jogos reais para escolher α.

Honestidade acima de tudo: o modelo já é **bem calibrado no histórico** e vence a
taxa-base em todas as fatias do walk-forward, então o α ótimo deve ser **≈0**. Se
for, aceitamos e relatamos que o modelo-base já é ótimo no walk-forward — não
forçamos melhora. O número que vale é o walk-forward, não os 20 jogos.

Usamos a **mesma fatia ≥1993** que seleciona o modelo (a mais parecida com 2026).
`base_rate` persistida = frequências de classe do **train completo** (em produção
não há "dobra"); a base por dobra serve só para tunar α sem vazamento.

In [40]:
# Busca em grade de alpha no walk-forward (fatia >=1993, a que seleciona o modelo).
# Coleta os folds UMA vez para o estimador final escolhido e varre a grade em cima
# (sem re-treinar por alpha). base por dobra (leak-free). Escolhe o argmin do RPS.
ALPHA_GRID = np.round(np.arange(0.0, 0.55, 0.05), 2)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    _folds_alpha = collect_folds(final_estimator, date_min="1993-01-01")

alpha_curve = pd.DataFrame(
    {"alpha": ALPHA_GRID,
     "rps_wf_1993": [rps_shrunk_over_folds(_folds_alpha, a) for a in ALPHA_GRID]}
)
rps_alpha0 = float(alpha_curve.loc[alpha_curve["alpha"] == 0.0, "rps_wf_1993"].iloc[0])
best_row = alpha_curve.loc[alpha_curve["rps_wf_1993"].idxmin()]
shrink_alpha = float(best_row["alpha"])
cv_rps_shrunk = float(best_row["rps_wf_1993"])
print("Curva RPS walk-forward (>=1993) por alpha:")
print(alpha_curve.round(6).to_string(index=False))
print(f"\n-> alpha* = {shrink_alpha:.2f} | RPS*(wf) = {cv_rps_shrunk:.6f} | "
      f"RPS(alpha=0) = {rps_alpha0:.6f} | ganho = {rps_alpha0 - cv_rps_shrunk:.6f}")
if shrink_alpha == 0.0:
    print("CONCLUSÃO: alpha*=0 — o modelo-base já é ótimo no walk-forward (sem shrinkage).")
elif (rps_alpha0 - cv_rps_shrunk) < 1e-3:
    print("CONCLUSÃO HONESTA: alpha pequeno com ganho DESPREZÍVEL (<1e-3) — o modelo-base "
          "já é ~ótimo no walk-forward; o shrinkage é cosmético, não uma melhora real.")

Curva RPS walk-forward (>=1993) por alpha:
 alpha  rps_wf_1993
  0.00     0.172523
  0.05     0.172434
  0.10     0.172641
  0.15     0.173144
  0.20     0.173942
  0.25     0.175035
  0.30     0.176424
  0.35     0.178109
  0.40     0.180090
  0.45     0.182366
  0.50     0.184937

-> alpha* = 0.05 | RPS*(wf) = 0.172434 | RPS(alpha=0) = 0.172523 | ganho = 0.000089
CONCLUSÃO HONESTA: alpha pequeno com ganho DESPREZÍVEL (<1e-3) — o modelo-base já é ~ótimo no walk-forward; o shrinkage é cosmético, não uma melhora real.


In [41]:
# base_rate persistida = frequências de classe do TRAIN COMPLETO, na ordem OUTCOMES.
# É o vetor aplicado em produção (nos 72 jogos não há "dobra"). A base por dobra
# acima serviu só para tunar alpha sem vazamento. Com alpha=0 ela é inócua.
base_rate = (
    y.value_counts(normalize=True).reindex(OUTCOMES).fillna(0.0).to_numpy(dtype=float)
)
assert np.isclose(base_rate.sum(), 1.0)
print("base_rate (home, draw, away):", base_rate.round(4).tolist())

# Salva a curva RPS x alpha em docs/train/figuras/ (registro versionado, como a EDA).
alpha_curve.round(6).to_csv(FIG_DIR / "shrinkage_alpha_walkforward.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(alpha_curve["alpha"], alpha_curve["rps_wf_1993"], "o-", color="#1f77b4")
ax.axvline(shrink_alpha, ls="--", color="#d62728",
           label=f"alpha* = {shrink_alpha:.2f}")
ax.set_xlabel("alpha (shrinkage para a taxa-base)")
ax.set_ylabel("RPS médio walk-forward (>=1993)")
ax.set_title("Shrinkage: RPS walk-forward por alpha")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "shrinkage_alpha_walkforward.png", dpi=120)
plt.close(fig)
print("curva alpha salva em", FIG_DIR / "shrinkage_alpha_walkforward.png")

base_rate (home, draw, away): [0.4901, 0.2274, 0.2825]
curva alpha salva em /Users/franklaercio/Workspace/paul-the-octopus/docs/train/figuras/shrinkage_alpha_walkforward.png


In [42]:
# Ajuste final: re-treina o estimador escolhido em TODO o train (com sample_weight).
final_pipe = clone(final_estimator)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    final_pipe.fit(X, y, clf__sample_weight=w.to_numpy(dtype=float))

# Smoke: predict_proba nas 72 linhas predict, JÁ com o shrinkage aplicado (a mesma
# função apply_shrinkage do 03/validate) -> (72, 3), sem NaN, soma ~1, em [0,1].
# Com shrink_alpha=0 isto reproduz as probabilidades cruas EXATAMENTE.
proba_raw = proba_in_outcome_order(final_pipe, predict[FEATURE_NAMES])
proba_predict = apply_shrinkage(proba_raw, base_rate, shrink_alpha)
assert proba_predict.shape == (len(predict), 3)
assert not np.isnan(proba_predict).any()
assert (proba_predict >= 0).all() and (proba_predict <= 1).all()
assert np.allclose(proba_predict.sum(axis=1), 1.0, atol=1e-6)
if shrink_alpha == 0.0:
    assert np.array_equal(proba_predict, proba_raw), "alpha=0 deve reproduzir o cru"
print("final_pipe fitted | predict proba:", proba_predict.shape,
      "| classes_:", list(final_pipe.classes_),
      "| shrink_alpha:", shrink_alpha)

final_pipe fitted | predict proba: (72, 3) | classes_: ['away', 'draw', 'home'] | shrink_alpha: 0.05


In [43]:
# Diagnóstico de calibração do modelo final (reliability one-vs-rest) na fatia
# de validação >=1993. Figura versionada em docs/train/figuras/ (registro, como a EDA).
# Probas/observados out-of-fold no último fold walk-forward da fatia >=1993.
_mask = _slice_since("1993-01-01")
_Xs, _ys = X_sorted[_mask], y_sorted[_mask]
_ws = w_sorted[_mask]
_tr, _te = list(TimeSeriesSplit(5).split(_Xs))[-1]
_m = clone(best_pipe)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    _m.fit(_Xs.iloc[_tr], _ys.iloc[_tr], clf__sample_weight=_ws.iloc[_tr].to_numpy(float))
_p = proba_in_outcome_order(_m, _Xs.iloc[_te]).ravel()
_o = onehot(_ys.iloc[_te]).ravel()
_edges = np.linspace(0, 1, 11)
_idx = np.clip(np.digitize(_p, _edges) - 1, 0, 9)
_mp, _of = [], []
for b in range(10):
    sel = _idx == b
    if sel.any():
        _mp.append(_p[sel].mean())
        _of.append(_o[sel].mean())
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="calibração perfeita")
ax.plot(_mp, _of, "o-", color="#1f77b4", label=f"{best_name} (out-of-fold, >=1993)")
ax.set_xlabel("Probabilidade prevista (one-vs-rest)")
ax.set_ylabel("Frequência observada")
ax.set_title("Diagrama de calibração — modelo final")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "calibracao_modelo.png", dpi=120)
plt.close(fig)
results.round(5).to_csv(FIG_DIR / "resultados_modelos.csv")
print("figura + tabela salvas em", FIG_DIR)

figura + tabela salvas em /Users/franklaercio/Workspace/paul-the-octopus/docs/train/figuras


In [44]:
# Serializa o bundle {'pipeline', 'meta'} (contrato §2.1 / §10.7).
# meta carrega a config de inferência que o 03/validate aplicam: shrink_alpha,
# base_rate e a flag de calibração. O shrinkage NÃO entra no Pipeline sklearn
# (depende da base e do mapeamento de classes); é aplicado por apply_shrinkage
# à saída de proba_in_outcome_order — a MESMA função no harness, no 03 e no validate.
meta = {
    "feature_names": list(FEATURE_NAMES),        # ordem exata de entrada do 03
    "classes": list(OUTCOMES),                   # ['home','draw','away'] (ordem do RPS)
    "model_name": model_name,                    # rótulo do escolhido (+ sufixo de calibração)
    "seed": SEED,                                # 42
    "sklearn_version": sklearn.__version__,      # guardrail de compatibilidade
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "calibrated": bool(use_calibration),         # CalibratedClassifierCV mantido? (delta de RPS)
    "shrink_alpha": float(shrink_alpha),         # alpha do shrinkage (argmin do RPS walk-forward)
    "base_rate": [float(v) for v in base_rate],  # taxa-base (home,draw,away) p/ o shrinkage
    "cv_rps": float(cv_rps_shrunk),              # RPS walk-forward >=1993 com o alpha (sanity)
}
bundle = {"pipeline": final_pipe, "meta": meta}
joblib.dump(bundle, MODELS / "model.joblib")
print("model.joblib salvo. meta:")
for k, v in meta.items():
    print(f"  {k}: {v}")

model.joblib salvo. meta:
  feature_names: ['is_neutral', 'confed_home', 'confed_away', 'elo_home', 'elo_away', 'elo_diff', 'rank_home', 'rank_away', 'rank_diff', 'points_diff', 'form_pts_home', 'form_pts_away', 'form_gf_home', 'form_gf_away', 'form_ga_home', 'form_ga_away', 'rest_days_home', 'rest_days_away', 'h2h_games', 'h2h_winrate_home', 'h2h_available', 'tournament_tier']
  classes: ['home', 'draw', 'away']
  model_name: logreg
  seed: 42
  sklearn_version: 1.7.1
  created_at: 2026-06-18T21:54:23+00:00
  calibrated: False
  shrink_alpha: 0.05
  base_rate: [0.49008097165991904, 0.22738866396761134, 0.28253036437246964]
  cv_rps: 0.17243444829771631


## E7 · Modelo de gols COMPANHEIRO (Poisson + Dixon-Coles)

Camada **ilustrativa** que adiciona **previsão de placar** — sem tocar no
classificador 1X2 acima (probabilidades e `shrink_alpha` **inalterados**; o 1X2
segue sendo a saída pontuada). Dois regressores de Poisson estimam os gols
esperados (λ) de mandante e visitante a partir dos **mesmos 22 preditores** da §8;
a matriz de placares `P(h,a)=Poisson(h;λ_home)·Poisson(a;λ_away)·τ(h,a)` (com o
termo de baixos placares de Dixon-Coles) dá `xg_home`/`xg_away` e o
`placar_provavel`.

**Escolha do regressor (medida no walk-forward ≥1993, leak-free):** o
`PoissonRegressor` linear (imputer+indicador+scaler+one-hot, como o ramo linear do
1X2) **venceu** o `HistGradientBoostingRegressor(loss="poisson")` em desvio Poisson
(1,2026 vs 1,2291) **e** em MAE de gols (0,9508 vs 0,9573). Apesar dos 41,5% sem
ranking favorecerem o HistGB NaN-native, o ramo linear com indicador de ausência
captura melhor o sinal de gols nesta escala — então adotamos o **`PoissonRegressor`**.
A escolha é re-medida abaixo (não hard-coded).

**ρ de Dixon-Coles:** ajustado por **verossimilhança 1D** nas 4 células de baixo
placar do treino (sem vazamento), com clamp na faixa válida e **fallback ρ≈−0,13**
(coerente com a correlação −0,145 da EDA) se o MLE degenerar. As funções de gols
(`predict_goals`, `score_matrix`, `most_likely_score`, `scores_from_lambdas`) são
importadas de `scripts.run_pipeline` — **fonte única** compartilhada por 02/03/
`validate_model`, sem derivações divergentes de λ/placar.

In [45]:
# Imports do componente de gols + alvos de placar (Int64 nullable -> float p/ regressão).
# As funções de gols (matriz DC, lambda, placar) vêm de scripts.run_pipeline: FONTE
# ÚNICA compartilhada por 02/03/validate_model. O 02 NÃO redefine essa matemática.
import sys  # noqa: E402

from scipy.optimize import minimize_scalar  # noqa: E402
from sklearn.linear_model import PoissonRegressor  # noqa: E402

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.run_pipeline import (  # noqa: E402
    RHO_BOUNDS,
    RHO_FALLBACK,
    matrix_to_1x2,
    most_likely_score,
    predict_goals,
    score_matrix,
)

y_home = train["home_score"].astype(float)   # alvo de gols do mandante (treino)
y_away = train["away_score"].astype(float)   # alvo de gols do visitante (treino)
y_home_sorted = y_home.iloc[_order].reset_index(drop=True)   # mesma ordem temporal do 1X2
y_away_sorted = y_away.iloc[_order].reset_index(drop=True)
print("alvos de gols:", y_home.shape, "| media home/away:",
      round(y_home.mean(), 3), round(y_away.mean(), 3))

# Candidato A — PoissonRegressor linear: MESMO pré-processamento do ramo 1X2 linear
# (imputer+indicador materializa o sinal pré-1993; scaler; one-hot de confed). alpha
# é a regularização L2 do GLM (pequena: deixa os dados falarem).
pipe_pois = Pipeline([
    ("pre", pre_lin),
    ("reg", PoissonRegressor(max_iter=1000, alpha=1e-4)),
])
# Candidato B — HistGB Poisson (NaN-native; categóricas nativas, como o 1X2 HistGB).
pipe_pois_hgb = Pipeline([
    ("to_cat", to_category),
    ("reg", HistGradientBoostingRegressor(
        loss="poisson", random_state=SEED, categorical_features="from_dtype")),
])

alvos de gols: (49400,) | media home/away: 1.757 1.182


In [46]:
# MLE 1D do rho de Dixon-Coles nas 4 células de baixo placar do TREINO (leak-free).
# Maximiza sum(log tau(h,a)) sobre os placares observados; clamp em RHO_BOUNDS e
# fallback fixo se o ótimo degenerar (tau<=0 / fora da faixa). rho<0 reforça
# empate/dependência negativa (corr -0.145 da EDA).
def fit_rho(lam_home, lam_away, hs, as_):
    lh = np.asarray(lam_home, float); la = np.asarray(lam_away, float)
    hs = np.asarray(hs); as_ = np.asarray(as_)

    def neg_ll(rho):
        t = np.ones(len(hs))
        t = np.where((hs == 0) & (as_ == 0), 1 - lh * la * rho, t)
        t = np.where((hs == 0) & (as_ == 1), 1 + lh * rho, t)
        t = np.where((hs == 1) & (as_ == 0), 1 + la * rho, t)
        t = np.where((hs == 1) & (as_ == 1), 1 - rho, t)
        if np.any(t <= 0):
            return 1e12
        return -np.sum(np.log(t))

    res = minimize_scalar(neg_ll, bounds=RHO_BOUNDS, method="bounded")
    rho = float(res.x)
    if not res.success or not (RHO_BOUNDS[0] <= rho <= RHO_BOUNDS[1]):
        return RHO_FALLBACK
    return rho


# Harness walk-forward dos REGRESSORES de gols (mesma disciplina temporal do 1X2:
# clone por fold, sample_weight no fit, avaliação sem peso, fatia >=1993). Treina um
# modelo por lado, mede desvio Poisson e MAE; e, de quebra, a sanidade do placar:
# acerto exato e o RPS do 1X2 IMPLÍCITO pela matriz DC vs. o classificador 1X2.
def evaluate_goals(estimator, n_splits=5, date_min="1993-01-01"):
    mask = _slice_since(date_min)
    Xs, ws = X_sorted[mask], w_sorted[mask]
    yhs, yas, ysl = y_home_sorted[mask], y_away_sorted[mask], y_sorted[mask]
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for tr_idx, te_idx in tscv.split(Xs):
        mh = clone(estimator); ma = clone(estimator)
        wtr = ws.iloc[tr_idx].to_numpy(dtype=float)
        mh.fit(Xs.iloc[tr_idx], yhs.iloc[tr_idx], reg__sample_weight=wtr)
        ma.fit(Xs.iloc[tr_idx], yas.iloc[tr_idx], reg__sample_weight=wtr)
        # rho ajustado SÓ no treino do fold (leak-free).
        lam_h_tr = np.clip(mh.predict(Xs.iloc[tr_idx]), 1e-6, None)
        lam_a_tr = np.clip(ma.predict(Xs.iloc[tr_idx]), 1e-6, None)
        rho = fit_rho(lam_h_tr, lam_a_tr,
                      yhs.iloc[tr_idx].to_numpy(), yas.iloc[tr_idx].to_numpy())
        lam_h = np.clip(mh.predict(Xs.iloc[te_idx]), 1e-6, None)
        lam_a = np.clip(ma.predict(Xs.iloc[te_idx]), 1e-6, None)
        yh_te = yhs.iloc[te_idx].to_numpy(); ya_te = yas.iloc[te_idx].to_numpy()
        # desvio Poisson (média dos dois lados) e MAE de gols.
        from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
        dev = (mean_poisson_deviance(yh_te, lam_h) + mean_poisson_deviance(ya_te, lam_a)) / 2
        mae = (mean_absolute_error(yh_te, lam_h) + mean_absolute_error(ya_te, lam_a)) / 2
        # placar/1X2 implícito por jogo (matriz DC), e acerto exato de placar.
        probs_g = np.empty((len(lam_h), 3)); exact = 0
        for i in range(len(lam_h)):
            matrix = score_matrix(lam_h[i], lam_a[i], rho)
            probs_g[i] = matrix_to_1x2(matrix)
            h_hat, a_hat = np.unravel_index(int(np.argmax(matrix)), matrix.shape)
            exact += int(h_hat == yh_te[i] and a_hat == ya_te[i])
        oh = onehot(ysl.iloc[te_idx])
        rows.append({
            "poisson_dev": dev, "mae_gols": mae, "acerto_exato": exact / len(lam_h),
            "rps_1x2_implicito": rps_mean(probs_g, oh), "rho": rho,
        })
    per_fold = pd.DataFrame(rows)
    return per_fold.mean().to_dict(), per_fold

In [47]:
# Seleção do regressor de gols pelo MENOR desvio Poisson walk-forward (>=1993). Mede
# os dois candidatos no MESMO split; reporta MAE, acerto exato e o RPS 1X2 implícito.
with warnings.catch_warnings():
    warnings.simplefilter("error")
    pois_lin_m, _ = evaluate_goals(pipe_pois)
    pois_hgb_m, _ = evaluate_goals(pipe_pois_hgb)

goals_table = pd.DataFrame(
    [{"modelo": "poisson_linear", **pois_lin_m},
     {"modelo": "histgb_poisson", **pois_hgb_m}]
).set_index("modelo")
print(goals_table.round(5))

goals_candidates = {"poisson_linear": pipe_pois, "histgb_poisson": pipe_pois_hgb}
goals_metrics = {"poisson_linear": pois_lin_m, "histgb_poisson": pois_hgb_m}
best_goals_name = min(goals_metrics, key=lambda k: goals_metrics[k]["poisson_dev"])
best_goals_pipe = goals_candidates[best_goals_name]
gm = goals_metrics[best_goals_name]
print(f"\n-> regressor de gols escolhido: {best_goals_name} "
      f"(menor desvio Poisson = {gm['poisson_dev']:.5f})")
print(f"   MAE gols={gm['mae_gols']:.4f} | acerto exato={gm['acerto_exato']*100:.2f}% "
      f"| RPS 1X2 implícito={gm['rps_1x2_implicito']:.5f}")
print(f"   RPS 1X2 classificador (referência) = {cv_rps_shrunk:.5f} "
      f"-> companheiro {'>=' if gm['rps_1x2_implicito'] >= cv_rps_shrunk else '<'} "
      "classificador (ilustrativo; não precisa vencer).")

                poisson_dev  mae_gols  acerto_exato  rps_1x2_implicito  \
modelo                                                                   
poisson_linear      1.20256   0.95076       0.13435            0.17229   
histgb_poisson      1.22912   0.95728       0.13361            0.17532   

                    rho  
modelo                   
poisson_linear -0.06078  
histgb_poisson -0.07507  

-> regressor de gols escolhido: poisson_linear (menor desvio Poisson = 1.20256)
   MAE gols=0.9508 | acerto exato=13.43% | RPS 1X2 implícito=0.17229
   RPS 1X2 classificador (referência) = 0.17243 -> companheiro < classificador (ilustrativo; não precisa vencer).


In [48]:
# Ajuste final do componente de gols: re-treina os dois regressores em TODO o train
# (com sample_weight) e ajusta o rho no train COMPLETO (1D MLE). Tudo determinístico.
home_model = clone(best_goals_pipe)
away_model = clone(best_goals_pipe)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    home_model.fit(X, y_home, reg__sample_weight=w.to_numpy(dtype=float))
    away_model.fit(X, y_away, reg__sample_weight=w.to_numpy(dtype=float))

lam_home_tr = np.clip(home_model.predict(X), 1e-6, None)
lam_away_tr = np.clip(away_model.predict(X), 1e-6, None)
rho = fit_rho(lam_home_tr, lam_away_tr,
              y_home.to_numpy(), y_away.to_numpy())
assert RHO_BOUNDS[0] <= rho <= RHO_BOUNDS[1]
print(f"rho (Dixon-Coles, MLE no train completo) = {rho:.4f} "
      f"(faixa {RHO_BOUNDS}; fallback {RHO_FALLBACK})")

goals_bundle = {
    "home_model": home_model,
    "away_model": away_model,
    "rho": float(rho),
    "feature_names": list(FEATURE_NAMES),   # mesma ordem do 1X2
    "model_name": best_goals_name,
}

rho (Dixon-Coles, MLE no train completo) = -0.0518 (faixa (-0.2, 0.2); fallback -0.13)


In [49]:
# Smoke nos 72 jogos predict: lambda finito/positivo e placar consistente com o 1X2
# JÁ pontuado (proba_predict, com shrinkage). O 1X2 NÃO muda — só restringe a região
# do argmax do placar (home: h>a; draw: h=a; away: h<a). Mesma derivação do 03.
lam_h_pred, lam_a_pred = predict_goals(goals_bundle, predict[FEATURE_NAMES])
assert np.isfinite(lam_h_pred).all() and (lam_h_pred > 0).all()
assert np.isfinite(lam_a_pred).all() and (lam_a_pred > 0).all()

outcomes_pred = [OUTCOMES[i] for i in np.argmax(proba_predict, axis=1)]
placar_provavel = []
for i, outcome in enumerate(outcomes_pred):
    matrix = score_matrix(lam_h_pred[i], lam_a_pred[i], rho)
    assert np.isclose(matrix.sum(), 1.0, atol=1e-6)
    h_hat, a_hat = most_likely_score(matrix, outcome)
    # invariante de consistência 1X2<->placar
    cons = (h_hat > a_hat) if outcome == "home" else \
           (h_hat < a_hat) if outcome == "away" else (h_hat == a_hat)
    assert cons, f"placar {h_hat}-{a_hat} contradiz 1X2 {outcome}"
    placar_provavel.append(f"{h_hat}-{a_hat}")

# Amostra: 3 jogos com xg e placar consistente (sanidade visual).
amostra = pd.DataFrame({
    "match_no": predict["match_no"].astype(int).to_numpy(),
    "1x2": outcomes_pred,
    "xg_home": np.round(lam_h_pred, 1),
    "xg_away": np.round(lam_a_pred, 1),
    "placar_provavel": placar_provavel,
}).head(3)
print("amostra (3 dos 72):")
print(amostra.to_string(index=False))

amostra (3 dos 72):
 match_no  1x2  xg_home  xg_away placar_provavel
        1 home      2.1      0.6             2-0
        2 away      1.2      1.3             0-1
        3 home      1.8      0.7             1-0


In [50]:
# Figura de diagnóstico do componente de gols: xg vs. gols reais por lado (>=1993,
# out-of-fold do último fold), registro versionado como as demais. Não vaza: usa o
# último split walk-forward da fatia >=1993, treino->predição fora da amostra.
_mask = _slice_since("1993-01-01")
_Xg, _wg = X_sorted[_mask], w_sorted[_mask]
_yh, _ya = y_home_sorted[_mask], y_away_sorted[_mask]
_tr, _te = list(TimeSeriesSplit(5).split(_Xg))[-1]
_mh, _ma = clone(best_goals_pipe), clone(best_goals_pipe)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    _mh.fit(_Xg.iloc[_tr], _yh.iloc[_tr], reg__sample_weight=_wg.iloc[_tr].to_numpy(float))
    _ma.fit(_Xg.iloc[_tr], _ya.iloc[_tr], reg__sample_weight=_wg.iloc[_tr].to_numpy(float))
_pred_h = np.clip(_mh.predict(_Xg.iloc[_te]), 1e-6, None)
_pred_a = np.clip(_ma.predict(_Xg.iloc[_te]), 1e-6, None)

# Calibração de gols por bins de xg: xg médio previsto vs. gols médios observados.
fig, ax = plt.subplots(figsize=(6, 5))
for pred, true, lab, color in [(_pred_h, _yh.iloc[_te].to_numpy(), "mandante", "#1f77b4"),
                               (_pred_a, _ya.iloc[_te].to_numpy(), "visitante", "#d62728")]:
    edges = np.linspace(0, max(3.0, pred.max()), 9)
    idx = np.clip(np.digitize(pred, edges) - 1, 0, len(edges) - 2)
    mp, mo = [], []
    for b in range(len(edges) - 1):
        sel = idx == b
        if sel.sum() >= 20:
            mp.append(pred[sel].mean()); mo.append(true[sel].mean())
    ax.plot(mp, mo, "o-", color=color, label=lab)
lim = 3.0
ax.plot([0, lim], [0, lim], "--", color="gray", label="calibração perfeita")
ax.set_xlabel("xg previsto (λ)"); ax.set_ylabel("gols observados (média)")
ax.set_title("Calibração do modelo de gols (out-of-fold, >=1993)")
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "calibracao_gols.png", dpi=120)
plt.close(fig)
goals_table.round(5).to_csv(FIG_DIR / "resultados_gols.csv")
print("figura + tabela de gols salvas em", FIG_DIR)

figura + tabela de gols salvas em /Users/franklaercio/Workspace/paul-the-octopus/docs/train/figuras


In [51]:
# Estende o bundle com a chave 'goals' e regrava model.joblib. O pipeline 1X2 e os
# campos 1X2 do meta (shrink_alpha, base_rate, classes...) ficam INTACTOS — só
# acrescentamos o componente de gols e a flag/rho/nome no meta para o 03/validate.
meta["has_goals"] = True                       # 03/skill detectam o componente
meta["goals_model_name"] = best_goals_name     # rótulo do regressor de gols
meta["goals_rho"] = float(rho)                  # rho Dixon-Coles persistido
meta["goals_poisson_dev"] = float(gm["poisson_dev"])   # desvio Poisson wf (sanity)

bundle["meta"] = meta
bundle["goals"] = goals_bundle
joblib.dump(bundle, MODELS / "model.joblib")
print("model.joblib regravado com componente de gols. Chaves do bundle:",
      sorted(bundle.keys()))
print("meta['has_goals'] =", meta["has_goals"],
      "| goals_model_name =", meta["goals_model_name"],
      "| goals_rho =", round(meta["goals_rho"], 4))

model.joblib regravado com componente de gols. Chaves do bundle: ['goals', 'meta', 'pipeline']
meta['has_goals'] = True | goals_model_name = poisson_linear | goals_rho = -0.0518


In [ ]:
# Ensemble 1X2 = classificador + 1X2 implícito de Dixon-Coles. Tuna o peso w do
# classificador por RPS walk-forward (>=1993; MESMA disciplina: clone por fold,
# sample_weight só no fit, rho ajustado só no treino do fold, avaliação sem peso). O
# shrinkage de produção (shrink_alpha) entra no RPS para casar com o 03/validate_model.
# Coleta os folds UMA vez (treina clf + gols por fold) e varre w em cima — leak-free.
def collect_ensemble_folds(clf_est, goals_est, n_splits=5, date_min="1993-01-01"):
    mask = _slice_since(date_min)
    Xs, ys, ws = X_sorted[mask], y_sorted[mask], w_sorted[mask]
    yhs, yas = y_home_sorted[mask], y_away_sorted[mask]
    folds = []
    for tr_idx, te_idx in TimeSeriesSplit(n_splits).split(Xs):
        wtr = ws.iloc[tr_idx].to_numpy(dtype=float)
        mc = clone(clf_est)
        mc.fit(Xs.iloc[tr_idx], ys.iloc[tr_idx], clf__sample_weight=wtr)
        proba_clf_fold = proba_in_outcome_order(mc, Xs.iloc[te_idx])
        mh, ma = clone(goals_est), clone(goals_est)
        mh.fit(Xs.iloc[tr_idx], yhs.iloc[tr_idx], reg__sample_weight=wtr)
        ma.fit(Xs.iloc[tr_idx], yas.iloc[tr_idx], reg__sample_weight=wtr)
        rho_f = fit_rho(np.clip(mh.predict(Xs.iloc[tr_idx]), 1e-6, None),
                        np.clip(ma.predict(Xs.iloc[tr_idx]), 1e-6, None),
                        yhs.iloc[tr_idx].to_numpy(), yas.iloc[tr_idx].to_numpy())
        lam_h = np.clip(mh.predict(Xs.iloc[te_idx]), 1e-6, None)
        lam_a = np.clip(ma.predict(Xs.iloc[te_idx]), 1e-6, None)
        proba_dc_fold = np.empty((len(lam_h), 3))
        for i in range(len(lam_h)):
            proba_dc_fold[i] = matrix_to_1x2(score_matrix(lam_h[i], lam_a[i], rho_f))
        rate = (ys.iloc[tr_idx].value_counts(normalize=True)
                .reindex(OUTCOMES).fillna(0.0).to_numpy())
        folds.append((proba_clf_fold, proba_dc_fold, rate, onehot(ys.iloc[te_idx])))
    return folds


def rps_ensemble_over_folds(folds, w_clf, alpha):
    """RPS médio entre folds do ensemble (w*clf + (1-w)*DC) + shrinkage — leak-free."""
    vals = []
    for proba_clf_fold, proba_dc_fold, rate, oh in folds:
        proba = w_clf * proba_clf_fold + (1.0 - w_clf) * proba_dc_fold
        vals.append(rps_mean(apply_shrinkage(proba, rate, alpha), oh))
    return float(np.mean(vals))


ENS_GRID = np.round(np.arange(0.0, 1.05, 0.05), 2)
with warnings.catch_warnings():
    warnings.simplefilter("error")
    _ens_folds = collect_ensemble_folds(best_pipe, best_goals_pipe)
ens_curve = pd.DataFrame({
    "w_clf": ENS_GRID,
    "rps_wf_1993": [rps_ensemble_over_folds(_ens_folds, w, shrink_alpha) for w in ENS_GRID],
})
rps_clf_only = float(ens_curve.loc[ens_curve["w_clf"] == 1.0, "rps_wf_1993"].iloc[0])
best_ens = ens_curve.loc[ens_curve["rps_wf_1993"].idxmin()]
ensemble_weight = float(best_ens["w_clf"])
cv_rps_ensemble = float(best_ens["rps_wf_1993"])
print("Curva RPS walk-forward (>=1993) por peso do classificador (w_clf):")
print(ens_curve.round(6).to_string(index=False))
print(f"\n-> w_clf* = {ensemble_weight:.2f} | RPS*(wf) = {cv_rps_ensemble:.6f} | "
      f"RPS(clf puro, w=1) = {rps_clf_only:.6f} | ganho = {rps_clf_only - cv_rps_ensemble:.6f}")
if ensemble_weight >= 1.0:
    print("CONCLUSAO: w*=1 - o classificador puro ja e otimo; o ensemble e inocuo.")
elif (rps_clf_only - cv_rps_ensemble) < 1e-4:
    print("CONCLUSAO HONESTA: ganho < 1e-4 - ensemble cosmetico; mantido por robustez "
          "(media de 2 preditores).")

# Curva versionada em docs/train/figuras/ (registro, como a curva de shrinkage).
ens_curve.round(6).to_csv(FIG_DIR / "ensemble_weight_walkforward.csv", index=False)
print("curva do ensemble salva em", FIG_DIR / "ensemble_weight_walkforward.csv")


In [ ]:
# Persiste o ensemble no meta e regrava o bundle. A proba 1X2 de PRODUCAO passa a ser
# apply_shrinkage(ensemble(clf, DC, w*), base_rate, shrink_alpha) - a MESMA do 03/
# validate_model. Recalcula proba_predict e revalida o smoke das 72 linhas.
meta["ensemble_weight"] = float(ensemble_weight)
meta["cv_rps_ensemble"] = float(cv_rps_ensemble)

# 1X2 implicito de Dixon-Coles nos 72 jogos (reusa lambda/rho do componente de gols).
proba_dc_pred = np.empty((len(predict), 3))
for i in range(len(lam_h_pred)):
    proba_dc_pred[i] = matrix_to_1x2(score_matrix(lam_h_pred[i], lam_a_pred[i], rho))
proba_ens_pred = ensemble_weight * proba_raw + (1.0 - ensemble_weight) * proba_dc_pred
proba_predict = apply_shrinkage(proba_ens_pred, base_rate, shrink_alpha)
assert proba_predict.shape == (len(predict), 3)
assert not np.isnan(proba_predict).any()
assert (proba_predict >= 0).all() and (proba_predict <= 1).all()
assert np.allclose(proba_predict.sum(axis=1), 1.0, atol=1e-6)

bundle["meta"] = meta
joblib.dump(bundle, MODELS / "model.joblib")
print("model.joblib regravado com ensemble | ensemble_weight =", meta["ensemble_weight"],
      "| RPS wf (ensemble) =", round(meta["cv_rps_ensemble"], 6))
print("p_draw medio nas 72: clf =", round(proba_raw[:, 1].mean(), 4),
      "-> final =", round(proba_predict[:, 1].mean(), 4),
      "| jogos com argmax=draw:", int((np.argmax(proba_predict, axis=1) == 1).sum()))
